# BraTS2020 — Extended Training of 3D U-Net (10 → 20 epochs)

**Mục đích:** Tài liệu hóa và hỗ trợ tái lập thí nghiệm kéo dài huấn luyện 3D U-Net từ checkpoint epoch 10 tới epoch 20, giữ nguyên phân chia dữ liệu, kiến trúc, loss, optimizer và cách đánh giá của baseline.

**Nguồn:** Tinh gọn từ notebook thực nghiệm 17 code cell đã chạy thành công trên Kaggle, ngày 23/10/2026. Notebook thực nghiệm gốc chứa output từng epoch; **bản này là phiên bản đọc, kiểm tra và tái lập có kiểm soát**, không giả vờ rằng các cell tinh gọn đã được chạy lại.

> **An toàn:** `RUN_TRAINING = False` theo mặc định. Chỉ bật khi cố ý tái lập thí nghiệm với GPU và checkpoint baseline. Không chạy lại huấn luyện để cập nhật các kết quả đã lưu. Dataset test gồm 56 bệnh nhân **không sử dụng** trong thí nghiệm này để lựa chọn checkpoint.

**Tham chiếu:** `notebooks/03_baseline_training.ipynb`, các module trong `src/`, `results/metrics/validation_epoch_10.csv` và `results/extended_20epochs/`. Checkpoint `.pt` lưu trong bản backup ngoài GitHub.

## Kết quả đã ghi nhận (không phải output tái chạy của notebook tinh gọn)

Trên **55 bệnh nhân validation nội bộ**, full-volume sliding-window inference, Dice macro theo bệnh nhân:

| Epoch | WT | TC | ET | Mean Dice |
|---:|---:|---:|---:|---:|
| 10 | 0.804653 | 0.665765 | 0.609885 | 0.693434 |
| 15 | 0.814428 | 0.679219 | 0.644341 | 0.712662 |
| **20** | **0.861699** | **0.743321** | **0.667613** | **0.757544** |

Chênh lệch Mean Dice **+0.064110** (khoảng **+9.25% tương đối**) ở epoch 20 so với epoch 10. Trong phân tích ghép cặp 55 bệnh nhân, Mean Dice tăng ở **48 ca**, giảm ở **7 ca**. Mô hình được chọn theo validation: **epoch 20**, `global_step=5137`.

Đây **không phải điểm test của mô hình 20 epoch**; mô hình 20 epoch chưa được đánh giá trên held-out test trong thí nghiệm được ghi nhận.

## 1. Cấu hình và công tắc thực thi

Notebook được tổ chức để đọc/kiểm tra an toàn. Khi `RUN_TRAINING=False`, không tải dữ liệu MRI, không tạo model, không bật optimizer hoặc sửa checkpoint. Các file kết quả đã commit lên nhánh thí nghiệm được dùng để xem lại kết quả.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tthanh81/brats2020-segmentation.git"
REPO_BRANCH = "experiment/unet3d-20epochs"
PROJECT_ROOT = Path("/kaggle/working/brats2020-extended-repo")
DATA_ROOT = Path("/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation")
BASELINE_BACKUP_ROOT = Path(
    "/kaggle/input/datasets/tthanhh811/brats2020-baseline-checkpoints-seed42"
)

RUN_NAME = "unet3d_extended_20epochs_seed42"
RUN_DIR = PROJECT_ROOT / "outputs" / "runs" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LATEST_CHECKPOINT = CHECKPOINT_DIR / "latest.pt"
BEST_CHECKPOINT = CHECKPOINT_DIR / "best.pt"
HISTORY_PATH = RUN_DIR / "training_history.csv"

SEED = 42
NUM_EPOCHS = 20
VAL_INTERVAL = 5
VAL_EPOCHS = (15, 20)
LEARNING_RATE = 1e-4
PATCH_SIZE = (96, 96, 96)
POS_PROB = 0.5
BATCH_SIZE = 1
NUM_WORKERS = 2

RUN_TRAINING = False    # Explicit opt-in; requires GPU and original datasets/checkpoint.
RUN_PAIRED_ANALYSIS = False  # Optional calculation from committed CSV, no GPU.

print("Training enabled:", RUN_TRAINING)
print("Paired analysis enabled:", RUN_PAIRED_ANALYSIS)
print("Output directory:", RUN_DIR)


## 2. Mở repository và xác minh phân chia cố định

Kaggle có `git` trong runtime. Trong một môi trường khác, bạn có thể clone thủ công rồi đặt `PROJECT_ROOT` tương ứng. **Không tự `git pull` khi đang huấn luyện**, để tránh thay đổi source code giữa chừng.

In [ ]:
import subprocess
import sys
import pandas as pd

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch",
         REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )
else:
    assert (PROJECT_ROOT / ".git").is_dir(), "PROJECT_ROOT is not a Git clone"

current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=PROJECT_ROOT, text=True,
).strip()
assert current_branch == REPO_BRANCH, (current_branch, REPO_BRANCH)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=PROJECT_ROOT, text=True,
).strip()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

manifest_path = PROJECT_ROOT / "splits/split_manifest_seed42.csv"
manifest = pd.read_csv(manifest_path)
assert manifest["subject_id"].is_unique
assert manifest["split"].value_counts().to_dict() == {
    "train": 257, "internal_validation": 55, "test": 56,
}
print("Branch:", current_branch, "| commit:", commit)
print("Manifest verified: train=257, validation=55, held-out test=56")


## 3. Kiểm tra checkpoint baseline trước khi (tùy chọn) huấn luyện

Nguồn khởi đầu là `latest.pt` epoch 10, cùng `best.pt` và `training_history.csv`. Phải có trạng thái model, AdamW và GradScaler; không khởi tạo huấn luyện lại từ trọng số ngẫu nhiên. Nếu `RUN_TRAINING=False`, cell chỉ thông báo vị trí dữ liệu đầu vào.

In [ ]:
import torch

SOURCE_LATEST = BASELINE_BACKUP_ROOT / "checkpoints/latest.pt"
SOURCE_BEST = BASELINE_BACKUP_ROOT / "checkpoints/best.pt"
SOURCE_HISTORY = BASELINE_BACKUP_ROOT / "results/training_history.csv"

if RUN_TRAINING:
    for src in (SOURCE_LATEST, SOURCE_BEST, SOURCE_HISTORY):
        assert src.is_file(), f"Attach the baseline backup dataset: {src}"
    source_checkpoint = torch.load(SOURCE_LATEST, map_location="cpu", weights_only=True)
    source_history = pd.read_csv(SOURCE_HISTORY)
    assert source_checkpoint["epoch"] == 10
    assert source_checkpoint["global_step"] == 2568
    assert abs(source_checkpoint["best_val_dice"] - 0.6934340347433116) < 1e-6
    assert len(source_history) == 10
    assert source_history["epoch"].tolist() == list(range(1, 11))
    for key in ("model_state_dict", "optimizer_state_dict", "scaler_state_dict"):
        assert key in source_checkpoint
    assert DATA_ROOT.is_dir(), "Attach the BraTS2020 MRI dataset"
    print("PASS: Epoch 10 source checkpoint and data available")
else:
    print("SKIP source checkpoint audit (RUN_TRAINING=False)")


## 4. Tái lập training (chỉ khi bật có chủ đích)

- MONAI 3D U-Net, 1,191,413 tham số; DiceCELoss, AdamW `1e-4`, patch `96³`, batch 1, xác suất tâm foreground `0.5`, AMP FP16.
- Resume **epoch 11** từ checkpoint epoch 10, giữ nguyên 257 training subjects; mốc validation **epoch 15 và epoch 20** trên 55 full volumes.
- `latest.pt` lưu sau mỗi epoch; `best.pt` chỉ cập nhật nếu **Mean Validation Dice** tăng. Việc lưu và ghi lịch sử diễn ra trước/sau validation sao cho có thể resume an toàn nếu phiên Kaggle gián đoạn.
- Thí nghiệm gốc đã hoàn thành; bật lại `RUN_TRAINING=True` là **tạo một lần chạy mới**, không phải tiếp tục thay đổi kết quả đã công bố. Để tránh ghi đè bản run gốc, hãy dùng `RUN_NAME` mới hoặc chạy trong một môi trường Kaggle trống.

RNG CPU/CUDA từ checkpoint được khôi phục khi resume, nhưng state của DataLoader workers và toàn bộ RNG của các thư viện khác không được lưu đầy đủ, nên không bảo đảm tái lập chính xác từng mini-batch.

In [ ]:
import os
import shutil
import math
import zipfile
import numpy as np
from torch.utils.data import DataLoader


def atomic_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    df.to_csv(temp, index=False)
    os.replace(temp, path)


def restore_torch_rng(ckpt):
    cpu_state, cuda_states = ckpt.get("torch_rng_state"), ckpt.get("cuda_rng_states")
    if cpu_state is None or cuda_states is None:
        raise RuntimeError("Missing RNG state in checkpoint")
    if len(cuda_states) != torch.cuda.device_count():
        raise RuntimeError("CUDA GPU count differs from saved checkpoint")
    torch.set_rng_state(cpu_state.clone())
    torch.cuda.set_rng_state_all([state.clone() for state in cuda_states])


def experiment_backup(run_dir, epoch):
    run_dir = Path(run_dir)
    target = Path("/kaggle/working") / f"brats2020_extended_epoch{epoch}_backup.zip"
    names = ["checkpoints/latest.pt", "checkpoints/best.pt", "training_history.csv"]
    names += [f"validation_epoch_{e:02d}.csv" for e in VAL_EPOCHS
              if (run_dir / f"validation_epoch_{e:02d}.csv").is_file()]
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for name in names:
            source = run_dir / name
            assert source.is_file(), source
            zf.write(source, arcname=name)
    with zipfile.ZipFile(target) as zf:
        assert zf.testzip() is None
    return target

print("Training helpers defined. No checkpoint modified.")


In [ ]:
# Training is opt-in. This cell makes no GPU training calls by default.
if RUN_TRAINING:
    from src.data.dataset import BraTSVolumeDataset
    from src.data.patches import RandomPatchDataset
    from src.models.unet3d import build_unet3d, build_loss
    from src.training.engine import train_one_epoch
    from src.training.checkpoints import load_checkpoint, save_checkpoint
    from src.evaluation.validation import validate_model

    assert torch.cuda.is_available(), "Enable GPU in Kaggle notebook settings"
    device = torch.device("cuda")
    train_df = manifest.loc[manifest["split"] == "train"].reset_index(drop=True)
    val_df = manifest.loc[manifest["split"] == "internal_validation"].reset_index(drop=True)
    assert len(train_df) == 257 and len(val_df) == 55

    train_ds = BraTSVolumeDataset(manifest=train_df, data_root=DATA_ROOT)
    val_ds = BraTSVolumeDataset(manifest=val_df, data_root=DATA_ROOT)
    patch_ds = RandomPatchDataset(base_dataset=train_ds,
                                  patch_size=PATCH_SIZE, pos_prob=POS_PROB)

    model = build_unet3d().to(device)
    assert sum(p.numel() for p in model.parameters()) == 1_191_413
    criterion = build_loss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    if LATEST_CHECKPOINT.exists():
        assert BEST_CHECKPOINT.is_file() and HISTORY_PATH.is_file(), (
            "Incomplete run directory. Review saved files before resuming."
        )
        checkpoint_path = LATEST_CHECKPOINT
        print("Resuming this experiment's existing output.")
    else:
        assert not BEST_CHECKPOINT.exists() and not HISTORY_PATH.exists(), (
            "Incomplete run directory. Do not overwrite prior results."
        )
        shutil.copy2(SOURCE_LATEST, LATEST_CHECKPOINT)
        shutil.copy2(SOURCE_BEST, BEST_CHECKPOINT)
        shutil.copy2(SOURCE_HISTORY, HISTORY_PATH)
        checkpoint_path = LATEST_CHECKPOINT
        print("Initialized new run from baseline epoch 10.")

    ckpt = load_checkpoint(checkpoint_path, model, optimizer, scaler,
                           map_location="cpu")
    history = pd.read_csv(HISTORY_PATH).to_dict("records")
    epoch_done = int(ckpt["epoch"])
    global_step = int(ckpt["global_step"])
    best_score = float(ckpt["best_val_dice"])
    assert 10 <= epoch_done <= 20
    assert len(history) == epoch_done
    assert [int(r["epoch"]) for r in history] == list(range(1, epoch_done + 1))
    assert int(history[-1]["global_step"]) == global_step

    training_config = dict(ckpt["config"])
    training_config.update({"num_epochs": NUM_EPOCHS, "resume_from_epoch": 10})

    def complete_validation(epoch):
        global best_score, history  # notebook-global values; used only when training enabled
        summary, details = validate_model(model=model, dataset=val_ds,
                                          device=device, roi_size=PATCH_SIZE,
                                          overlap=0.25)
        assert summary["num_subjects"] == 55 and len(details) == 55
        assert details["subject_id"].is_unique
        assert set(details["subject_id"]) == set(val_df["subject_id"])
        score = float(summary["mean_dice"])
        assert math.isfinite(score) and 0 <= score <= 1
        val_path = RUN_DIR / f"validation_epoch_{epoch:02d}.csv"
        atomic_csv(details, val_path)
        if score > best_score:
            best_score = score
            save_checkpoint(BEST_CHECKPOINT, model, optimizer, scaler,
                            epoch, global_step, best_score, training_config)
        history[-1].update({"val_WT": summary["WT"],
                            "val_TC": summary["TC"], "val_ET": summary["ET"],
                            "val_mean_dice": score})
        atomic_csv(pd.DataFrame(history), HISTORY_PATH)
        save_checkpoint(LATEST_CHECKPOINT, model, optimizer, scaler,
                        epoch, global_step, best_score, training_config)
        print(f"Epoch {epoch}: validation Mean Dice={score:.6f} | best={best_score:.6f}")

    # If interrupted after training the validation epoch, validate before proceeding.
    if epoch_done in VAL_EPOCHS and pd.isna(history[-1]["val_mean_dice"]):
        complete_validation(epoch_done)
        ckpt = torch.load(LATEST_CHECKPOINT, map_location="cpu", weights_only=True)

    if epoch_done < NUM_EPOCHS:
        restore_torch_rng(ckpt)
        loader = DataLoader(patch_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True,
                            persistent_workers=True, prefetch_factor=2)
        assert len(loader) == 257
        for epoch in range(epoch_done + 1, NUM_EPOCHS + 1):
            result = train_one_epoch(model, loader, criterion, optimizer,
                                     scaler, device, epoch, log_interval=25)
            assert result["optimizer_steps"] + result["skipped_steps"] == 257
            assert math.isfinite(float(result["mean_loss"]))
            global_step += result["optimizer_steps"]
            history.append({"epoch": epoch,
                            "train_loss": result["mean_loss"],
                            "epoch_time_seconds": result["epoch_time_seconds"],
                            "global_step": global_step,
                            "positive_patches": result["positive_patches"],
                            "tumor_patches": result["tumor_patches"],
                            "val_WT": None, "val_TC": None,
                            "val_ET": None, "val_mean_dice": None})
            save_checkpoint(LATEST_CHECKPOINT, model, optimizer, scaler,
                            epoch, global_step, best_score, training_config)
            atomic_csv(pd.DataFrame(history), HISTORY_PATH)
            if epoch in VAL_EPOCHS:
                complete_validation(epoch)
            print("Saved:", experiment_backup(RUN_DIR, epoch))
        print("PASS: 20-epoch training complete. Check best.pt by validation.")
    else:
        print("No additional epochs required; saved run already reached 20.")
else:
    print("SKIP training — RUN_TRAINING=False (default).")


## 5. Kiểm tra bảng kết quả đã lưu trong repository

Sau khi đưa `results/extended_20epochs/` lên GitHub, bảng này hoạt động **không cần GPU hoặc checkpoint**. Nếu notebook đang được chạy trước lúc commit, các CSV chưa tồn tại trong clone thì cell sẽ chỉ thông báo, không tự tạo kết quả giả.

In [ ]:
METRIC_ROOT = PROJECT_ROOT / "results/extended_20epochs/metrics"
COMPARISON_PATH = METRIC_ROOT / "validation_comparison_10_15_20.csv"
SUMMARY_PATH = METRIC_ROOT / "final_validation_summary.json"

if COMPARISON_PATH.is_file():
    comparison = pd.read_csv(COMPARISON_PATH)
    assert set(comparison["epoch"].tolist()) == {10, 15, 20}
    assert abs(float(comparison.loc[comparison["epoch"] == 20,
                                    "mean_dice"].iloc[0]) - 0.757544) < 1e-5
    print(comparison.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
else:
    print("Committed metrics not found yet:", COMPARISON_PATH)
    print("Add results/extended_20epochs/metrics to the experiment branch.")

if SUMMARY_PATH.is_file():
    import json
    with SUMMARY_PATH.open(encoding="utf-8") as f:
        result_summary = json.load(f)
    assert result_summary["best_epoch"] == 20
    print("Selected best epoch:", result_summary["best_epoch"])
    print("Mean validation Dice:", result_summary["best_val_dice"])


## 6. Phân tích ghép cặp theo bệnh nhân (tùy chọn)

So sánh cùng 55 bệnh nhân để tránh hiểu sai rằng Mean Dice tăng chỉ do vài giá trị ngoại lệ. Khi `RUN_PAIRED_ANALYSIS=False`, có thể xem lại số liệu đã lưu trong `results/extended_20epochs/metrics/validation_change_summary.csv` mà không tính lại.

In [ ]:
if RUN_PAIRED_ANALYSIS:
    import matplotlib.pyplot as plt

    val_paths = {
        10: PROJECT_ROOT / "results/metrics/validation_epoch_10.csv",
        15: METRIC_ROOT / "validation_epoch_15.csv",
        20: METRIC_ROOT / "validation_epoch_20.csv",
    }
    df_by_epoch = {}
    for epoch, path in val_paths.items():
        assert path.is_file(), f"Missing {path}"
        frame = pd.read_csv(path)
        assert len(frame) == 55 and frame["subject_id"].is_unique
        df_by_epoch[epoch] = frame.set_index("subject_id")

    ids = sorted(df_by_epoch[10].index)
    assert all(set(df.index) == set(ids) for df in df_by_epoch.values())
    paired = pd.DataFrame(index=ids)
    paired.index.name = "subject_id"
    for epoch in (10, 15, 20):
        for region in ("WT", "TC", "ET"):
            paired[f"{region}_{epoch}"] = df_by_epoch[epoch].loc[ids, region]
        paired[f"mean_{epoch}"] = paired[[f"{r}_{epoch}" for r in ("WT", "TC", "ET")]].mean(axis=1)

    change_rows = []
    for region in ("WT", "TC", "ET", "mean"):
        delta = paired[f"{region}_20"] - paired[f"{region}_10"]
        paired[f"delta_{region}"] = delta
        change_rows.append({"region": region, "mean_change": delta.mean(),
                            "median_change": delta.median(),
                            "improved": int((delta > 1e-8).sum()),
                            "worsened": int((delta < -1e-8).sum()),
                            "unchanged": int((delta.abs() <= 1e-8).sum())})
    changes = pd.DataFrame(change_rows)
    assert int(changes.loc[changes.region == "mean", "improved"].iloc[0]) == 48
    print(changes.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

    # Save optional rerun results away from the committed reference artifacts.
    analysis_dir = Path("/kaggle/working/brats2020-paired-analysis")
    analysis_dir.mkdir(parents=True, exist_ok=True)
    paired.reset_index().to_csv(analysis_dir / "validation_paired_deltas.csv", index=False)
    changes.to_csv(analysis_dir / "validation_change_summary.csv", index=False)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.boxplot([paired[f"delta_{r}"] for r in ("WT", "TC", "ET", "mean")],
               tick_labels=("WT", "TC", "ET", "Mean"), showmeans=True)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_ylabel("Dice change (epoch 20 - epoch 10)")
    ax.set_title("Paired validation improvement — 55 subjects")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(analysis_dir / "validation_paired_deltas.png", dpi=180)
    plt.show()
    plt.close(fig)
else:
    print("SKIP paired analysis. See saved CSV and figure in results/extended_20epochs/.")


## 7. Các kết luận và giới hạn

**Kết quả đã quan sát:** Mean Validation Dice tăng từ 0.693434 (epoch 10) lên 0.757544 (epoch 20). Trên 55 validation subjects, 48 có Mean Dice cải thiện và 7 suy giảm. WT và TC tăng đáng kể; ET vẫn là vùng khó nhất.

**Giới hạn:** Đây là **một lần chia dữ liệu cố định (seed 42), một lần kéo dài huấn luyện** và kết quả tính trên cùng validation split. Không có bằng chứng từ notebook này về external validation, repeated seeds, confidence intervals hoặc test của checkpoint 20 epoch. Held-out test đã được đánh giá trước đây cho baseline 10 epoch, nhưng **không dùng kết quả đó để chọn checkpoint mới**.

**Artifacts và nguồn chứng cứ:**
- Nhật ký thí nghiệm gốc 17 cell, với output đã thực thi: lưu riêng (Kaggle Quick Save / bản `.ipynb` tải về).
- Mã reusable: `src/data/`, `src/models/`, `src/training/`, `src/evaluation/`.
- Bảng và biểu đồ: `results/extended_20epochs/metrics/`, `results/extended_20epochs/figures/`.
- Checkpoints `best.pt` / `latest.pt`: backup Kaggle riêng, **không commit GitHub**.
- Các giá trị trong bảng đầu notebook là kết quả lịch sử đã xác minh bằng output thí nghiệm, không phải kết quả được tạo lại bởi bản tinh gọn này.